# analysis.diversity-correlations

In this analysis, we want to quantify:

- Whether Bacteria diversity is correlated with Virus diversity
- Whether Bacteria diversity is correlated with Plant diversity
- Whether habitat or disturbance play a role in these interactions.

To run this analysis, we will use R `lme4` package instead of our regular Python libraries. We will only use Python to create the dataframes. 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from daforfer import DaforferDB
import matplotlib.pyplot as plt
import networkx as nx
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

/home/bcz/miniconda3/envs/miripvir25/lib/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


┌──────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│   name   │                                                        description                                                        │
│ varchar  │                                                          varchar                                                          │
├──────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ TableS1  │ Table S1: Library sites and context                                                                                       │
│ TableS2  │ This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc. │
│ TableS3  │ Site-level diversity and number of cooccurring virus-bacteria                                                             │
│ TableS4  │ Habitat-level diversity and 

## Registering data

As everything was computed elswhere, we only need to save the results into the database. 

In [ ]:
site_level_regression = pd.read_csv("output/regression.site-level.csv", index_col=0)
db.save_dataframe(
    site_level_regression, 
    "T_ADAllOrganismsSite", 
    description="Binomial GLMs and random mixed effect models, obtained in R, at site level"
)

site_level_regression

Saved T_ADAllOrganismsSite to db.2025-10-27


,Model,Estimate,StdError,ZValue,PValue,DFres,LogLik
1,Model 1 - species_richness_vir,0.030261,0.011978,2.526466,0.011522,21,-79.561507
2,Model 2 - species_richness_plant,0.042152,0.012171,3.463258,0.000534,21,-79.105665
3,Model 3 - species_richness_plant,0.029144,0.011989,2.430900,0.015061,20,-77.323148
4,Model 3 - species_richness_vir,0.020985,0.011692,1.794905,0.072669,20,-77.323148
5,Model 4 - species_richness_vir + (1|habitat),0.030261,0.012281,2.464104,0.013736,19,-79.561507
6,Model 5 - species_richness_plant + (1|habitat),0.042152,0.014411,2.924958,0.003445,19,-79.105665
7,Model 6 - species_richness_plant + (1|habitat),0.029144,0.013692,2.128529,0.033293,18,-77.323148
8,Model 6 - species_richness_vir + (1|habitat),0.020985,0.011324,1.853150,0.063861,18,-77.323148
9,Model 7 - species_richness_vir + (1|dist),0.030261,0.012281,2.464092,0.013736,19,-79.561507
10,Model 8 - species_richness_plant + (1|dist),0.042152,0.014411,2.924952,0.003445,19,-79.105665


: 

In [11]:
site_level_regression['m'] = site_level_regression['Model'].apply(lambda x: x.split(" - ")[0])
Table1 = site_level_regression.query(
    'm == "Model 2" or m == "Model 5" or m == "Model 8"' 
).copy().drop(columns=['m'])
si.save_dataframe(
    Table1, "Table1",
    description="PAB diversity-plant diversity GLM models"
)
Table1

Saved Table1 to si.2025-10-27


,Model,Estimate,StdError,ZValue,PValue,DFres,LogLik
2,Model 2 - species_richness_plant,0.042152,0.012171,3.463258,0.000534,21,-79.105665
6,Model 5 - species_richness_plant + (1|habitat),0.042152,0.014411,2.924958,0.003445,19,-79.105665
10,Model 8 - species_richness_plant + (1|dist),0.042152,0.014411,2.924952,0.003445,19,-79.105665


In [12]:
site_level_regression
si.save_dataframe(
    site_level_regression, "TableS6",
    description="PAB diversity-plant diversity GLM models, including virus fits"
)
site_level_regression


Saved TableS6 to si.2025-10-27


,Model,Estimate,StdError,ZValue,PValue,DFres,LogLik,m
1,Model 1 - species_richness_vir,0.030261,0.011978,2.526466,0.011522,21,-79.561507,Model 1
2,Model 2 - species_richness_plant,0.042152,0.012171,3.463258,0.000534,21,-79.105665,Model 2
3,Model 3 - species_richness_plant,0.029144,0.011989,2.430900,0.015061,20,-77.323148,Model 3
4,Model 3 - species_richness_vir,0.020985,0.011692,1.794905,0.072669,20,-77.323148,Model 3
5,Model 4 - species_richness_vir + (1|habitat),0.030261,0.012281,2.464104,0.013736,19,-79.561507,Model 4
6,Model 5 - species_richness_plant + (1|habitat),0.042152,0.014411,2.924958,0.003445,19,-79.105665,Model 5
7,Model 6 - species_richness_plant + (1|habitat),0.029144,0.013692,2.128529,0.033293,18,-77.323148,Model 6
8,Model 6 - species_richness_vir + (1|habitat),0.020985,0.011324,1.853150,0.063861,18,-77.323148,Model 6
9,Model 7 - species_richness_vir + (1|dist),0.030261,0.012281,2.464092,0.013736,19,-79.561507,Model 7
10,Model 8 - species_richness_plant + (1|dist),0.042152,0.014411,2.924952,0.003445,19,-79.105665,Model 8


In [13]:
habitat_level_regression = pd.read_csv("output/regression.habitat-level.csv", index_col=0)
db.save_dataframe(
    habitat_level_regression, 
    "T_ADAllOrganismsHab", 
    description="Binomial GLMs and random mixed effect models, obtained in R, at habitat level"
)
habitat_level_regression

Saved T_ADAllOrganismsHab to db.2025-10-27


,Model,Estimate,StdError,ZValue,PValue,DFres,LogLik
1,Model 1 - species_richness_vir,0.010974,0.003638,3.016718,0.002555,2,-15.279452
2,Model 2 - species_richness_plant,0.001213,0.009818,0.123523,0.901693,2,-17.515836
3,Model 3 - species_richness_plant,0.004729,0.004966,0.952367,0.340911,1,-14.855155
4,Model 3 - species_richness_vir,0.011456,0.003215,3.563698,0.000366,1,-14.855155


In [14]:
db.conn.close()
si.conn.close()